

# 📈 IDX Event-Driven Stock Analysis & Scoring System

**Oleh: Muhammad Erico Ricardo**

## 🌟 Ringkasan Proyek

Proyek ini adalah sistem analisis saham otomatis berbasis *event-driven* yang memproses keterbukaan informasi dari Bursa Efek Indonesia (IDX) dan berita media finansial. Sistem ini tidak hanya mengumpulkan data, tetapi juga melakukan pembobotan (*Weighted Scoring*)  untuk mengidentifikasi emiten yang berpotensi mengalami pergerakan harga signifikan akibat aksi korporasi.

Inspirasi strategi: **Andry Hakim** (Expert Investor).

-----

## 🎯 Latar Belakang & Masalah

Dalam pasar modal Indonesia, berita mengenai aksi korporasi seringkali menjadi katalis utama kenaikan harga saham. Namun, volume berita harian sangat besar sehingga sulit bagi investor untuk menyaring mana berita yang benar-benar berdampak besar secara manual.

Proyek ini mendeteksi 8 katalis utama:

  * **High Impact:** Akuisisi, Pengambilalihan, Perubahan Pengendali.
  * **Medium-High Impact:** Right Issue, Penambahan Modal, Inbreng.
  * **Medium Impact:** Kerja sama strategis/Afiliasi, Ekspansi.

-----

## 🛠️ Tech Stack

  * **Language:** Python
  * **Web Scraping:** BeautifulSoup, Selenium (untuk konten dinamis IDX).

-----


---
# 🏗️ Phase 1: Automated Data Acquisition - IDX Official Announcements
Proses ini adalah fondasi utama untuk mendapatkan data primer langsung dari bursa. Karena situs keterbukaan informasi IDX bersifat dinamis (menggunakan JavaScript untuk memuat tabel), sistem ini menggunakan **Selenium WebDriver**.

**Logika Utama Scraping:**
1. **Dynamic Interaction:** Script melakukan simulasi interaksi pengguna dengan memilih filter tanggal dan jenis pengumuman secara otomatis.
2. **Handling Pagination:** Sistem secara rekursif menelusuri setiap halaman tabel pengumuman untuk memastikan tidak ada data emiten yang terlewat dalam rentang waktu yang ditentukan.
3. **Data Extraction:** Mengekstrak elemen kunci seperti kode emiten (Ticker), judul pengumuman, dan tautan dokumen PDF untuk analisis lebih lanjut.

## 1.1. Data Source 1 (Primary) : Melakukan scraping dari keterbukaan informasi IDX selama 3 hari kebelakang.

In [1]:
# Import Keperluan web scraping
import re
import time
import json
import pandas as pd
import os
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from urllib.parse import urljoin
from urllib.parse import quote

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By

In [2]:
# 1) CONFIG web scraping

BASE_URL = "https://www.idx.co.id/id/berita/pengumuman/" # URL dasar untuk pengumuman berita
CUT_OFF_DAYS = 3 # batas waktu untuk berita yang akan diambil (dalam hari)
TARGET_MAX_PAGES = 100  # pengaman loop agar tidak berjalan selamanya jika terjadi masalah pada website

cutoff_datetime = datetime.now() - timedelta(days=CUT_OFF_DAYS) # menghitung tanggal cutoff berdasarkan waktu saat ini

options = webdriver.ChromeOptions() 
options.add_argument("--start-maximized")
# options.add_argument("--headless=new")  # aktifkan kalau mau tanpa browser

browser = webdriver.Chrome(options=options)
wait = WebDriverWait(browser, 15)

In [3]:
# 2) HELPER

def normalize_text(text): # normalisasi teks: hapus spasi berlebih, ganti newline/tab dengan spasi
    if text is None:
        return None
    return re.sub(r"\s+", " ", text).strip()


def parse_date(date_text): # parsing tanggal dengan format yang umum digunakan di Indonesia
    """
    Contoh input:
    '18 Apr 2026 20:17:03'
    """
    if not date_text:
        return None

    text = normalize_text(date_text).replace(",", "")

    for fmt in ("%d %b %Y %H:%M:%S", "%d %b %Y"):
        try:
            return datetime.strptime(text, fmt)
        except ValueError:
            pass

    return None


def scroll_to_bottom(): # Fungsi untuk scroll ke bawah sampai tidak ada penambahan tinggi halaman lagi
    """
    Scroll sampai tinggi halaman berhenti bertambah.
    """
    last_height = browser.execute_script("return document.body.scrollHeight")

    while True:
        browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        new_height = browser.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height


def is_button_disabled(button): # Cek apakah tombol "Muat Lebih Banyak" dalam keadaan disabled
    disabled_attr = button.get_attribute("disabled")
    aria_disabled = button.get_attribute("aria-disabled")
    class_name = button.get_attribute("class") or ""

    if disabled_attr is not None:
        return True
    if aria_disabled == "true":
        return True
    if "disabled" in class_name.lower():
        return True
    return False


def extract_kode(judul): # Ekstrak kode dari judul berita
    """
    Handle format:
    [SKRN ], [SKYB ], [FAST], [KEJU]
    """
    judul = str(judul)
    match = re.search(r"\[\s*([A-Z]{4})\s*\]", judul)
    if match:
        return match.group(1)
    return None


def parse_local_number(num_str): # Parsing angka dengan format lokal Indonesia (menggunakan titik sebagai pemisah ribuan dan koma sebagai pemisah desimal)
    """
    Parse angka lokal Indonesia.
    Contoh:
    2,5 -> 2.5
    1.250 -> 1250
    10,75 -> 10.75
    """
    if num_str is None:
        return None

    s = str(num_str).strip()
    s = s.replace(" ", "")

    if "," in s and "." in s:
        # contoh: 1.234,56 -> 1234.56
        s = s.replace(".", "").replace(",", ".")
    elif "," in s:
        # contoh: 2,5 -> 2.5
        s = s.replace(".", "").replace(",", ".")
    else:
        # contoh: 1.250 -> 1250
        s = s.replace(".", "")

    try:
        return float(s)
    except ValueError:
        return None


def extract_transaction_value(text):
    """
    Deteksi nilai transaksi dari teks.
    Output dalam rupiah (int) jika ditemukan.

    Contoh:
    - Rp 2,5 triliun -> 2500000000000
    - Rp 500 miliar -> 500000000000
    - Rp 75 juta -> 75000000
    """
    if not text:
        return None

    raw = normalize_text(text).lower()
    raw = raw.replace("rupiah", "rp")
    raw = raw.replace("idr", "rp")

    patterns = [
        (r"(?:rp)\s*([\d\.,]+)\s*(triliun|t)\b", 1_000_000_000_000),
        (r"(?:rp)\s*([\d\.,]+)\s*(miliar|m)\b", 1_000_000_000),
        (r"(?:rp)\s*([\d\.,]+)\s*(juta|jt)\b", 1_000_000),
        (r"([\d\.,]+)\s*(triliun|t)\b", 1_000_000_000_000),
        (r"([\d\.,]+)\s*(miliar|m)\b", 1_000_000_000),
        (r"([\d\.,]+)\s*(juta|jt)\b", 1_000_000),
    ]

    for pattern, multiplier in patterns:
        match = re.search(pattern, raw)
        if match:
            number_str = match.group(1)
            value = parse_local_number(number_str)
            if value is not None:
                return int(value * multiplier)

    return None


def categorize_size(value): # Kategorikan ukuran berdasarkan nilai transaksi
    if value is None or pd.isna(value):
        return "Unknown"
    if value >= 1_000_000_000_000:
        return "Huge (>=1T)"
    if value >= 100_000_000_000:
        return "Large (>=100B)"
    if value >= 10_000_000_000:
        return "Medium (>=10B)"
    return "Small (<10B)"


def size_score(value): # Menghitung skor berdasarkan ukuran transaksi
    if value is None or pd.isna(value):
        return 0
    if value >= 1_000_000_000_000:
        return 3
    if value >= 100_000_000_000:
        return 2
    if value >= 10_000_000_000:
        return 1
    return 0

In [4]:
# 3) Kata kunci untuk klasifikasi berita dan scoring

KEYWORDS_SKIP = [
    "laporan keuangan",
    "public expose",
    "rups tahunan",
    "rups",
    "dividen",
]

# Urutan penting: Right Issue didahulukan agar tidak dobel hitung
KEYWORDS_RIGHT_ISSUE = [
    "right issue",
    "hm etd",
    "hmetd",
    "pmhmetd",
    "pm hm etd",
    "pmhm etd",
]

KEYWORDS_UTAMA = {
    "Akuisisi": ["akuisisi", "mengakuisisi"],
    "Pengambilalihan": ["pengambilalihan", "takeover"],
    "Perubahan Pengendali": [
        "perubahan pengendali",
        "pengendali baru",
        "perubahan kepemilikan saham yang menyebabkan perubahan pengendalian",
    ],
    "Inbreng / Setoran Aset": [
        "inbreng", "setoran aset", "penyertaan aset", 
        "penyertaan dalam bentuk selain kas", "penambahan modal non kas", 
        "setoran non kas", "konversi aset menjadi saham",
    ],
    "Transaksi Material": ["transaksi material"],
    "Afiliasi": ["afiliasi"],
    "Right Issue": ["right issue", "rights issue", "hm etd", "hmetd", "pmhmetd", "pm hm etd", "pmhm etd", "penerbitan saham baru"], 
    "Penambahan Modal": ["penambahan modal", "private placement"],
}

KEYWORDS_ABU = [
    "kerja sama",
    "kerjasama",
    "ekspansi",
    "investasi",
]

SCORE_MAP = {
    "Akuisisi": 3,
    "Pengambilalihan": 3,
    "Perubahan Pengendali": 3,
    "Right Issue": 2,
    "Penambahan Modal": 2,
    "Inbreng / Setoran Aset": 3,
    "Transaksi Material": 2,
    "Afiliasi": 1,
    "Keyword Abu-Abu": 1,
}


def classify_news(judul): 
    judul_lower = str(judul).lower()

    if any(word in judul_lower for word in KEYWORDS_SKIP):
        return None

    categories = []

    # Right Issue diprioritaskan agar tidak dobel dengan Penambahan Modal
    if any(word in judul_lower for word in KEYWORDS_RIGHT_ISSUE):
        categories.append("Right Issue")
    elif any(word in judul_lower for word in ["penambahan modal", "pmhm etd", "pmhmetd"]):
        categories.append("Penambahan Modal")

    for key, values in KEYWORDS_UTAMA.items():
        if key in ["Penambahan Modal"]:  # sudah ditangani di atas
            continue
        if any(v in judul_lower for v in values):
            categories.append(key)

    if any(word in judul_lower for word in KEYWORDS_ABU):
        categories.append("Keyword Abu-Abu")

    categories = sorted(set(categories))
    return ", ".join(categories) if categories else None


def calculate_score(kategori):
    """
    Skor dasar per kategori.
    Bonus +1 jika ada 2 kategori atau lebih.
    """
    if not kategori or pd.isna(kategori):
        return 0

    items = [x.strip() for x in str(kategori).split(",") if x.strip()]
    base = sum(SCORE_MAP.get(item, 0) for item in items)
    bonus = 1 if len(items) >= 2 else 0
    return base + bonus

In [5]:
# 4) Proses Scraping IDX

browser.get(BASE_URL)
wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
time.sleep(3)

scraped_rows = []
seen_titles = set()
page_num = 1
stop_scraping = False

while not stop_scraping and page_num <= TARGET_MAX_PAGES:
    print(f"\n📄 Memproses halaman {page_num} ...")

    scroll_to_bottom()
    time.sleep(3)

    title_elements = browser.find_elements(By.CSS_SELECTOR, ".f-m-20.title")

    if not title_elements:
        print("⚠️ Tidak menemukan judul berita di halaman ini.")
    else:
        for idx, title_el in enumerate(title_elements, start=1):
            try:
                title_text = normalize_text(title_el.text)
                if not title_text:
                    continue

                if title_text in seen_titles:
                    continue
                seen_titles.add(title_text)

                try:
                    container = title_el.find_element(By.XPATH, "./ancestor::*[.//time][1]")
                    time_el = container.find_element(By.TAG_NAME, "time")
                    date_text = normalize_text(time_el.text)
                except Exception:
                    date_text = None

                parsed_dt = parse_date(date_text)

                if parsed_dt and parsed_dt < cutoff_datetime:
                    print(f"🛑 Ditemukan data lebih tua dari H-{CUT_OFF_DAYS}: {parsed_dt}")
                    stop_scraping = True
                    break

                scraped_rows.append(
                    {
                        "judul": title_text,
                        "tanggal": date_text,
                        "tanggal_dt": parsed_dt.strftime("%Y-%m-%d %H:%M:%S") if parsed_dt else None,
                    }
                )

                print(f"✅ {idx}. {title_text} | {date_text}")

            except Exception as error:
                print(f"❌ Gagal ambil item ke-{idx}: {error}")

    if stop_scraping:
        break

    try:
        next_button = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "button.btn-arrow.--next"))
        )

        if is_button_disabled(next_button):
            print("⏹ Tombol next page sudah tidak aktif. Selesai.")
            break

        browser.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_button)
        time.sleep(1)
        browser.execute_script("arguments[0].click();", next_button)

        print("➡️ Pindah ke halaman berikutnya ...")
        page_num += 1
        time.sleep(3)

    except Exception as error:
        print(f"❌ Tidak bisa klik next page: {error}")
        break

browser.quit()


📄 Memproses halaman 1 ...
✅ 1. Laporan Harian atas Nilai Aktiva Bersih dan Komposisi Portofolio [XILV ] | 27 Apr 2026 22:18:33
✅ 2. Rencana Pengalihan Saham Treasuri Hasil Pembelian Kembali Saham [INDY ] | 27 Apr 2026 21:54:34
✅ 3. Penyampaian Laporan Tahunan [TPMA ] | 27 Apr 2026 21:51:05
✅ 4. Laporan Harian atas Nilai Aktiva Bersih dan Komposisi Portofolio [R-ABFII ] | 27 Apr 2026 21:46:20
✅ 5. Penambahan modal tanpa HMETD - Rencana Pelaksanaan Penambahan Modal Tanpa HMETD [DGNS ] | 27 Apr 2026 21:37:12
✅ 6. Penyampaian Bukti Iklan Informasi Penambahan Modal tanpa HMETD [SAME ] | 27 Apr 2026 21:22:37
✅ 7. Penyampaian Bukti Iklan Keterbukaan Informasi Rencana PMTHMETD dalam rangka Pelaksanaan Program Kepemilikan Saham Manajemen dan Karyawan (Program MESOP) [SAME ] | 27 Apr 2026 21:21:33
✅ 8. Penyampaian Bukti Iklan Pemberitahuan RUPS [SAME ] | 27 Apr 2026 21:19:28
✅ 9. Laporan Harian atas Nilai Aktiva Bersih dan Komposisi Portofolio [XDIF ] | 27 Apr 2026 21:17:53
✅ 10. Penyampaian La

In [6]:
# 5) Menyimpan Raw Data hasil scraping ke CSV untuk analisis selanjutnya

df = pd.DataFrame(scraped_rows)
df.to_csv("idx_raw.csv", index=False, encoding="utf-8-sig")

print(f"\n💾 Raw data tersimpan: idx_raw.csv")
print(f"📊 Total raw data: {len(df)}")


💾 Raw data tersimpan: idx_raw.csv
📊 Total raw data: 329


In [7]:
# 6) Data Cleaning, Filtering, Ekstraksi Kode Saham, dan Deteksi Nilai Transaksi

# Mengambil kode saham dan kategori berita
df["kode"] = df["judul"].apply(extract_kode)
df["kategori"] = df["judul"].apply(classify_news)

# gabungkan teks sumber untuk deteksi nilai transaksi
df["teks_sumber"] = df["judul"].fillna("")

# Cek berapa data yang tidak memiliki kode saham
missing_kode = df["kode"].isnull().sum()
print(f"❗ Data tanpa kode saham: {missing_kode}")

# Filter data yang memiliki kode saham dan kategori berita
df = df[df["kode"].notnull()].copy()
filtered_df = df[df["kategori"].notnull()].copy()

print(f"✅ Total setelah filter keyword: {len(filtered_df)}")

# Ekstrak nilai transaksi, kategorikan ukuran, dan hitung skor
filtered_df["nilai_transaksi"] = filtered_df["teks_sumber"].apply(extract_transaction_value)
filtered_df["size_category"] = filtered_df["nilai_transaksi"].apply(categorize_size)
filtered_df["score_size"] = filtered_df["nilai_transaksi"].apply(size_score)
filtered_df["score_keyword"] = filtered_df["kategori"].apply(calculate_score)
filtered_df["final_score"] = filtered_df["score_keyword"] + filtered_df["score_size"]

# Simpan hasil akhir ke CSV
filtered_df.to_csv("idx_filtered.csv", index=False, encoding="utf-8-sig")

print("\n📊 Sample hasil setelah scoring:")
print(filtered_df[["kode", "judul", "kategori", "nilai_transaksi", "size_category", "final_score"]].head(10))

❗ Data tanpa kode saham: 21
✅ Total setelah filter keyword: 11

📊 Sample hasil setelah scoring:
     kode                                              judul  \
4    DGNS  Penambahan modal tanpa HMETD - Rencana Pelaksa...   
5    SAME  Penyampaian Bukti Iklan Informasi Penambahan M...   
6    SAME  Penyampaian Bukti Iklan Keterbukaan Informasi ...   
21   CDIA  Penyampaian Bukti Iklan KETERBUKAAN INFORMASI ...   
39   BAUT                         Transaksi Afiliasi [BAUT ]   
41   BUVA                         Transaksi Afiliasi [BUVA ]   
46   LINK  Penyampaian Informasi rencana Transaksi materi...   
111  EPAC  Negosiasi rencana pengambilalihan Perseroan [E...   
135  CDIA  KETERBUKAAN INFORMASI PT CHANDRA DAYA INVESTAS...   
301  SAME  Keterbukaan Informasi Penambahan Modal Tanpa H...   

                      kategori nilai_transaksi size_category  final_score  
4                  Right Issue            None       Unknown            2  
5                  Right Issue            None 

In [8]:
# 7) Analisis dan Summary per Emiten

company_summary = (
    filtered_df
    .groupby("kode")
    .agg(
        total_score=("final_score", "sum"),
        jumlah_berita=("judul", "count"),
        rata_rata_score=("final_score", "mean"),
        max_score=("final_score", "max"),
        nilai_transaksi_rata2=("nilai_transaksi", "mean"),
        nilai_transaksi_maks=("nilai_transaksi", "max"),
        judul_list=("judul", list),
        kategori_list=("kategori", lambda x: list(x)),
        tanggal_terbaru=("tanggal_dt", "max"),
    )
    .reset_index()
    .sort_values(by=["total_score", "jumlah_berita"], ascending=False)
)

company_summary.to_csv("idx_company_summary.csv", index=False, encoding="utf-8-sig")

top_10 = company_summary.head(10).copy()
top_10.to_csv("idx_top_10_emitten.csv", index=False, encoding="utf-8-sig")

print("\n🔥 Top 10 Emiten berdasarkan score:")
print(top_10[["kode", "total_score", "jumlah_berita", "nilai_transaksi_maks", "tanggal_terbaru"]])


print("✅ Selesai.")


🔥 Top 10 Emiten berdasarkan score:
   kode  total_score  jumlah_berita nilai_transaksi_maks      tanggal_terbaru
6  SAME            8              4                  NaN  2026-04-27 21:22:37
2  CDIA            6              2                  NaN  2026-04-27 20:49:47
4  EPAC            3              1                  NaN  2026-04-27 17:51:30
3  DGNS            2              1                  NaN  2026-04-27 21:37:12
5  LINK            2              1                  NaN  2026-04-27 19:25:55
0  BAUT            1              1                  NaN  2026-04-27 19:44:05
1  BUVA            1              1                  NaN  2026-04-27 19:34:37
✅ Selesai.


## 1.2 Data Source 2 (Secondary) : Mengambil Berita mendalam dari *Bloomberg Technoz dan IDX Channel* dengan menggunakan *Keyword* emiten dengan score tertinggi dari proses data source 1.

### 1.2.1 Mengambil berita dari IDX Channel 

In [9]:
# 1) Konfigurasi awal untuk web scraping

CUT_OFF_DAYS = 90 # Mengambil berita dalam 90 hari terakhir
cutoff_date = datetime.now() - timedelta(days=CUT_OFF_DAYS)

browser = webdriver.Chrome()
browser.maximize_window()

In [10]:
# 2) Fungsi bantu untuk ekstraksi teks artikel, deteksi nilai transaksi, dan klasifikasi berita

def extract_article_text(page_url):
    browser.get(page_url)
    time.sleep(2)

    soup = BeautifulSoup(browser.page_source, "html.parser")

    # Gunakan fallback selector seperti di Bloomberg
    container = None
    candidate_selectors = [
        "div#content-news", 
        "div.detail-berita", 
        "div.text-body", 
        "div[class*='text-body']", 
        "article"
    ]

    for sel in candidate_selectors:
        container = soup.select_one(sel)
        if container:
            break

    if container is None:
        return "", soup, [] # Gagal ambil body, tapi judul masih bisa dipakai nanti

    # REMOVE NOISE
    for tag in container.select("style, script, noscript, iframe, img, .baca-juga-new, .paging"):
        tag.decompose()

    paragraphs = []
    for p in container.find_all("p"):
        text = normalize_text(p.get_text(" ", strip=True))
        if not text:
            continue
        if any(x in text.lower() for x in ["baca juga", "simak juga"]):
            continue
        paragraphs.append(text)

    full_text = "\n".join(paragraphs)
    tags = extract_tags(soup)

    return full_text, soup, tags

In [11]:
# 3) Mengambil tag terkait dari halaman berita untuk membantu klasifikasi dan analisis lebih lanjut

def extract_tags(soup):
    tags = []
    for a in soup.select("a[href*='/tag/']"):
        m = re.search(r"/tag/([A-Z0-9]+)", a.get("href", ""))
        if m:
            tags.append(m.group(1))
    return list(set(tags))

In [12]:
# 4) Memeriksa apakah berita relevan dengan kode saham tertentu berdasarkan judul, teks, dan tag yang terkait

# Fungsi ini menggabungkan judul, teks, dan tag menjadi satu string besar, lalu memeriksa apakah kode saham muncul di dalamnya. Ini membantu menangkap berita yang mungkin tidak menyebutkan kode secara eksplisit di judul tetapi relevan berdasarkan isi dan tag.
def is_relevant(code, title, text, tags):
    merged = f"{title} {text} {' '.join(tags)}".upper()
    return code.upper() in merged

# Deteksi apakah berita lebih banyak membahas tentang kondisi pasar secara umum (noise) daripada tentang emiten tertentu
def is_market_noise(text):
    keywords = ["ihsg", "indeks", "market cap", "perdagangan"]
    t = text.lower()
    return sum(1 for k in keywords if k in t) >= 2

In [13]:
# 5) Deteksi nilai transaksi dari teks berita, dengan fokus pada format yang umum digunakan di Indonesia (misalnya "Rp 2,5 triliun", "Rp 500 miliar", "Rp 75 juta"). Fungsi ini akan mengembalikan nilai dalam rupiah sebagai integer jika ditemukan, atau None jika tidak ada nilai yang terdeteksi.
def parse_number(s):
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except:
        return None


def extract_value(text):
    text = text.lower()

    patterns = [
        (r"([\d\.,]+)\s*(triliun|t)", 1e12),
        (r"([\d\.,]+)\s*(miliar|m)", 1e9),
        (r"([\d\.,]+)\s*(juta|jt)", 1e6),
    ]

    for p, mult in patterns:
        m = re.search(p, text)
        if m:
            val = parse_number(m.group(1))
            if val:
                return int(val * mult)
    return None


def size_score(val):
    if not val: return 0
    if val >= 1e12: return 3
    if val >= 1e11: return 2
    if val >= 1e10: return 1
    return 0

In [14]:
# 6) Klasifikasi berita berdasarkan kata kunci utama dan abu-abu, serta menghitung skor berdasarkan kategori dan nilai transaksi
def detect_narrative(text):
    t = text.lower()
    found = []

    for k, v in KEYWORDS_UTAMA.items():
        if any(x in t for x in v):
            found.append(k)

    if any(x in t for x in KEYWORDS_ABU):
        found.append("Keyword Abu-Abu")

    return list(dict.fromkeys(found))


In [15]:
# 7) Mengambil daftar kode saham dari top 10 emiten untuk fokus analisis lebih lanjut
top10 = pd.read_csv("idx_top_10_emitten.csv")
codes = top10["kode"].tolist()

In [16]:
# 8) Mencari berita terkait kode saham

search_results = []

for i, code in enumerate(codes, 1):
    print(f"\n🔎 [{i}/{len(codes)}] {code}")

    browser.get(f"https://www.idxchannel.com/search?search={code}")
    time.sleep(3)

    prev_count = 0

    while True:
        soup = BeautifulSoup(browser.page_source, "html.parser")
        items = soup.select("h2.list-berita-baru")

        print(f"📊 Items: {len(items)}")

        stop_flag = False

        for h2 in items:
            a = h2.find("a")
            if not a: continue

            title = normalize_text(a.text)
            url = a["href"]

            date_el = h2.find_next("span", class_="mh-clock")
            date = parse_date(date_el.text if date_el else "")

            if date and date < cutoff_date:
                stop_flag = True
                break

            search_results.append({
                "kode": code,
                "judul": title,
                "url": url
            })

        if stop_flag:
            break

        if len(items) >= 20:
            print("⏹ Stop: di atas 20 items, kemungkinan sudah tidak relevan lagi")
            break

        if len(items) == prev_count:
            print("⏹ Stop: stagnan")
            break

        prev_count = len(items)

        try:
            btn = browser.find_element(By.ID, "next-row")
            browser.execute_script("arguments[0].click();", btn)
            print("🔽 Load more")
            time.sleep(2)
        except:
            break

search_df = pd.DataFrame(search_results).drop_duplicates("url")
search_df.to_csv("search.csv", index=False)


🔎 [1/7] SAME
📊 Items: 9
🔽 Load more
📊 Items: 18
🔽 Load more
📊 Items: 27
⏹ Stop: di atas 20 items, kemungkinan sudah tidak relevan lagi

🔎 [2/7] CDIA
📊 Items: 9
🔽 Load more
📊 Items: 18
🔽 Load more
📊 Items: 27
⏹ Stop: di atas 20 items, kemungkinan sudah tidak relevan lagi

🔎 [3/7] EPAC
📊 Items: 9
🔽 Load more
📊 Items: 18
🔽 Load more
📊 Items: 27
⏹ Stop: di atas 20 items, kemungkinan sudah tidak relevan lagi

🔎 [4/7] DGNS
📊 Items: 9
🔽 Load more
📊 Items: 18
🔽 Load more
📊 Items: 18
⏹ Stop: stagnan

🔎 [5/7] LINK
📊 Items: 9
🔽 Load more
📊 Items: 18
🔽 Load more
📊 Items: 18
⏹ Stop: stagnan

🔎 [6/7] BAUT
📊 Items: 9
🔽 Load more
📊 Items: 9
⏹ Stop: stagnan

🔎 [7/7] BUVA
📊 Items: 9
🔽 Load more
📊 Items: 18
🔽 Load more
📊 Items: 27
⏹ Stop: di atas 20 items, kemungkinan sudah tidak relevan lagi


In [17]:
# 9) Proses detail setiap berita yang ditemukan, ekstrak teks, tag, klasifikasi, dan scoring
articles = []

for i, row in search_df.iterrows():
    print(f"\n📄 {i+1}/{len(search_df)} {row['kode']}")

    try:
        text, soup, tags = extract_article_text(row["url"])
        title = normalize_text(soup.find("h1").text if soup.find("h1") else "")

        if not is_relevant(row["kode"], title, text, tags):
            print("⏭ skip relevance")
            continue

        if is_market_noise(text):
            print("⚠️ market noise skip")
            continue


        # GABUNGKAN JUDUL DAN TEKS SEBELUM DETEKSI
        full_content_for_analysis = f"{title} {text}"
        
        val = extract_value(full_content_for_analysis)
        nar = detect_narrative(full_content_for_analysis)

        score = sum(SCORE_MAP.get(x, 0) for x in nar)
        score += size_score(val)


        articles.append({
            "kode": row["kode"],
            "judul": title,
            "nilai": val,
            "narasi": ",".join(nar),
            "score": score
        })

        print(f"✅ score: {score}")

    except Exception as e:
        print("❌ error", e)

browser.quit()

article_df = pd.DataFrame(articles)
article_df.to_csv("articles.csv", index=False)


📄 1/143 SAME
⏭ skip relevance

📄 2/143 SAME
⏭ skip relevance

📄 3/143 SAME
✅ score: 0

📄 4/143 SAME
⏭ skip relevance

📄 5/143 SAME
⏭ skip relevance

📄 6/143 SAME
⏭ skip relevance

📄 7/143 SAME
⏭ skip relevance

📄 8/143 SAME
⏭ skip relevance

📄 9/143 SAME
⏭ skip relevance

📄 19/143 SAME
⏭ skip relevance

📄 20/143 SAME
⏭ skip relevance

📄 21/143 SAME
⏭ skip relevance

📄 22/143 SAME
⏭ skip relevance

📄 23/143 SAME
⏭ skip relevance

📄 24/143 SAME
⏭ skip relevance

📄 25/143 SAME
⏭ skip relevance

📄 26/143 SAME
⏭ skip relevance

📄 27/143 SAME
⏭ skip relevance

📄 46/143 SAME
⏭ skip relevance

📄 47/143 SAME
⏭ skip relevance

📄 48/143 SAME
⏭ skip relevance

📄 49/143 SAME
⏭ skip relevance

📄 50/143 SAME
⏭ skip relevance

📄 51/143 SAME
⏭ skip relevance

📄 52/143 SAME
⏭ skip relevance

📄 53/143 SAME
⏭ skip relevance

📄 54/143 SAME
⏭ skip relevance

📄 55/143 CDIA
⏭ skip relevance

📄 56/143 CDIA
⏭ skip relevance

📄 57/143 CDIA
⏭ skip relevance

📄 58/143 CDIA
⏭ skip relevance

📄 59/143 CDIA
⏭ skip r

### 1.2.2 Mengambil Berita dari Bloomberg technoz

In [18]:
# 1) Konfigurasi awal untuk web scraping

CUT_OFF_DAYS = 90 # Mengambil berita dalam 90 hari terakhir
cutoff_date = datetime.now() - timedelta(days=CUT_OFF_DAYS)

browser = webdriver.Chrome()
browser.maximize_window()

In [19]:
# 1) Konfigurasi awal untuk web scraping
def extract_search_results(code):


    results = []
    page_num = 1
    prev_count = 0

    while True:
        if page_num == 1:
            url = f"https://www.bloombergtechnoz.com/search?query={quote(code)}"
        else:
            url = f"https://www.bloombergtechnoz.com/search?query={quote(code)}&type=berita&pagenum={page_num}"

        print(f"🔎 [{code}] buka halaman {page_num}: {url}")
        browser.get(url)
        time.sleep(2.5)

        soup = BeautifulSoup(browser.page_source, "html.parser")

        # Setiap card biasanya berupa <a href="/detail-news/...">
        items = soup.select("a[href*='/detail-news/']")

        print(f"📊 items found: {len(items)}")

        if len(items) == 0:
            break

        for a in items:
            href = a.get("href", "")
            if not href:
                continue

            # URL bisa absolut atau relatif
            if href.startswith("/"):
                url_full = "https://www.bloombergtechnoz.com" + href
            else:
                url_full = href

            title_el = a.select_one("h2.title.margin-bottom-xs")
            title = normalize_text(title_el.get_text(" ", strip=True)) if title_el else ""

            # Ambil waktu dari h6 atau span cl-gray
            time_el = a.select_one("h6 .cl-gray") or a.select_one("span.cl-gray")
            date = parse_date(time_el.get_text(" ", strip=True) if time_el else "")

            # Kalau tidak ada title, skip
            if not title:
                continue

            # Stop bila sudah melewati cutoff, asumsi hasil search urut dari terbaru ke lama
            if date and date < cutoff_date:
                print("⏹ stop: sudah lewat cutoff")
                return pd.DataFrame(results).drop_duplicates("url")

            results.append({
                "kode": code.upper(),
                "judul": title,
                "url": url_full,
                "date": date
            })

        # Jika jumlah item stagnan, stop
        if len(items) == prev_count:
            print("⏹ stop: stagnan")
            break

        prev_count = len(items)
        page_num += 1

    return pd.DataFrame(results).drop_duplicates("url")


In [20]:
# 2) Clean text
def extract_article_text(page_url):
    """
    Bloomberg Technoz struktur detail page bisa berbeda-beda.
    Kita pakai beberapa fallback selector agar lebih tahan banting.
    """
    browser.get(page_url)
    time.sleep(2.5)

    soup = BeautifulSoup(browser.page_source, "html.parser")

    # Fallback selector konten artikel
    container = None
    candidate_selectors = [
        "div.detail-news",
        "div.article-content",
        "div.content-detail",
        "article",
        "div[class*='detail']",
        "div[class*='article']",
        "main",
    ]

    for sel in candidate_selectors:
        container = soup.select_one(sel)
        if container:
            break

    if container is None:
        return "", soup, []

    # Remove noise
    for tag in container.select("style, script, noscript, iframe, img, figure, aside"):
        tag.decompose()

    paragraphs = []
    for p in container.find_all("p"):
        text = normalize_text(p.get_text(" ", strip=True))
        if not text:
            continue

        if any(x in text.lower() for x in ["baca juga", "simak juga", "klik untuk"]):
            continue

        paragraphs.append(text)

    full_text = "\n".join(paragraphs)

    tags = extract_tags(soup)

    return full_text, soup, tags


In [21]:
# 3) Extract tags
def extract_tags(soup):
    """
    Jika Bloomberg Technoz memiliki tag/label/hastag di detail page,
    ambil secara longgar. Jika tidak ada, return list kosong.
    """
    tags = []

    # Fallback umum untuk tag/news category
    for a in soup.select("a[href*='/tag/'], a[href*='category'], a[class*='tag']"):
        txt = normalize_text(a.get_text(" ", strip=True))
        if txt and len(txt) <= 40:
            tags.append(txt.upper())

    # unik
    return list(set(tags))

In [22]:
# 4) Mencari Berita Relevan
search_results = []

for i, code in enumerate(codes, 1):
    print(f"\n🔎 [{i}/{len(codes)}] {code}")

    try:
        df_code = extract_search_results(code)
        if len(df_code) > 0:
            search_results.append(df_code)
    except Exception as e:
        print("❌ error search:", e)

if len(search_results) > 0:
    search_df = pd.concat(search_results, ignore_index=True).drop_duplicates("url")
else:
    search_df = pd.DataFrame(columns=["kode", "judul", "url", "date"])

search_df.to_csv("search.csv", index=False)


🔎 [1/7] SAME
🔎 [SAME] buka halaman 1: https://www.bloombergtechnoz.com/search?query=SAME
📊 items found: 38
🔎 [SAME] buka halaman 2: https://www.bloombergtechnoz.com/search?query=SAME&type=berita&pagenum=2
📊 items found: 35
🔎 [SAME] buka halaman 3: https://www.bloombergtechnoz.com/search?query=SAME&type=berita&pagenum=3
📊 items found: 35
⏹ stop: stagnan

🔎 [2/7] CDIA
🔎 [CDIA] buka halaman 1: https://www.bloombergtechnoz.com/search?query=CDIA
📊 items found: 45
🔎 [CDIA] buka halaman 2: https://www.bloombergtechnoz.com/search?query=CDIA&type=berita&pagenum=2
📊 items found: 45
⏹ stop: stagnan

🔎 [3/7] EPAC
🔎 [EPAC] buka halaman 1: https://www.bloombergtechnoz.com/search?query=EPAC
📊 items found: 36
🔎 [EPAC] buka halaman 2: https://www.bloombergtechnoz.com/search?query=EPAC&type=berita&pagenum=2
📊 items found: 35
🔎 [EPAC] buka halaman 3: https://www.bloombergtechnoz.com/search?query=EPAC&type=berita&pagenum=3
📊 items found: 35
⏹ stop: stagnan

🔎 [4/7] DGNS
🔎 [DGNS] buka halaman 1: https://w

In [23]:
# 5) Detail Berita, Klasifikasi, dan Scoring
articles = []

for i, row in search_df.iterrows():
    print(f"\n📄 {i+1}/{len(search_df)} {row['kode']}")

    try:
        text, soup, tags = extract_article_text(row["url"])
        title = normalize_text(soup.find("h1").get_text(" ", strip=True) if soup.find("h1") else row["judul"])

        if not is_relevant(row["kode"], title, text, tags):
            print("⏭ skip relevance")
            continue

        if is_market_noise(text):
            print("⚠️ market noise skip")
            continue

        val = extract_value(text)
        nar = detect_narrative(text)

        score = sum(SCORE_MAP.get(x, 0) for x in nar)
        score += size_score(val)

        articles.append({
            "kode": row["kode"],
            "judul": title,
            "url": row["url"],
            "nilai": val,
            "narasi": ",".join(nar),
            "score": score
        })

        print(f"✅ score: {score}")

    except Exception as e:
        print("❌ error", e)

browser.quit()





📄 1/54 SAME
⏭ skip relevance

📄 2/54 SAME
✅ score: 0

📄 3/54 SAME
✅ score: 0

📄 4/54 CDIA
✅ score: 1

📄 5/54 CDIA
✅ score: 4

📄 6/54 CDIA
✅ score: 4

📄 7/54 CDIA
⏭ skip relevance

📄 8/54 CDIA
✅ score: 4

📄 9/54 CDIA
✅ score: 4

📄 10/54 CDIA
✅ score: 8

📄 11/54 CDIA
⚠️ market noise skip

📄 12/54 CDIA
⏭ skip relevance

📄 13/54 CDIA
✅ score: 4

📄 14/54 CDIA
✅ score: 5

📄 15/54 CDIA
⚠️ market noise skip

📄 16/54 CDIA
✅ score: 1

📄 17/54 CDIA
✅ score: 7

📄 18/54 CDIA
✅ score: 4

📄 19/54 CDIA
⚠️ market noise skip

📄 20/54 CDIA
✅ score: 4

📄 21/54 CDIA
✅ score: 1

📄 22/54 CDIA
✅ score: 1

📄 23/54 CDIA
⚠️ market noise skip

📄 24/54 EPAC
✅ score: 3

📄 25/54 LINK
✅ score: 3

📄 26/54 LINK
✅ score: 0

📄 27/54 LINK
⏭ skip relevance

📄 28/54 LINK
✅ score: 4

📄 29/54 LINK
✅ score: 3

📄 30/54 LINK
⏭ skip relevance

📄 31/54 LINK
✅ score: 0

📄 32/54 LINK
✅ score: 0

📄 33/54 LINK
⏭ skip relevance

📄 34/54 LINK
✅ score: 0

📄 35/54 LINK
⏭ skip relevance

📄 36/54 LINK
⏭ skip relevance

📄 37/54 LINK
✅ score

In [24]:
# menambahkan data article_df ke baris baru di pada file articles.csv, jika file sudah ada, jika tidak buat baru
if os.path.exists("articles.csv"):
    existing_df = pd.read_csv("articles.csv")
    combined_df = pd.concat([existing_df, pd.DataFrame(articles)], ignore_index=True)
    combined_df.to_csv("articles.csv", index=False)
else:
    article_df = pd.DataFrame(articles)
    article_df.to_csv("articles.csv", index=False)

## 1.3 Menggabungkan Scoring dan mencari top 10 Emiten dengan score tertinggi.

In [25]:
# 10. Menggabungkan skor berita dengan data top 10 emiten, menghitung skor gabungan, dan menyimpan hasil akhir
if len(article_df) > 0:
    agg = article_df.groupby("kode").agg(
        news_score=("score", "sum"),
        news_count=("judul", "count"),
        max_value=("nilai", "max"),
        judul_list=("judul", list)
    ).reset_index()
else:
    agg = pd.DataFrame(columns=["kode", "news_score", "news_count", "max_value", "judul_list"])

final = top10.merge(agg, on="kode", how="left")
final["news_score"] = final["news_score"].fillna(0)
final["news_count"] = final["news_count"].fillna(0)
final["max_value"] = final["max_value"].fillna(0)
final["combined"] = final["total_score"] + final["news_score"]

final = final.sort_values("combined", ascending=False)
final.to_csv("final.csv", index=False)

print("\n🚀 DONE")


🚀 DONE


---
# 🤖 Phase 2: AI Context Injection & Prompt Engineering
Masalah utama AI saat ini adalah *Knowledge Cutoff*. Dengan menyuntikkan hasil *scoring* dan ringkasan berita terbaru ke dalam *prompt*, kita memberikan "mata" bagi AI untuk melihat kondisi pasar terkini.

**Struktur Prompt:**
* **Data Input:** Top 10 emiten dengan skor tertinggi dan ringkasan berita terkait.
* **Task:** AI diminta untuk melakukan validasi fundamental dan memberikan opini teknikal singkat berdasarkan konteks data yang baru saja di-*scrape*.
* **Output:** Rekomendasi yang lebih akurat dan relevan dengan kondisi pasar hari ini.

In [26]:
def build_master_prompt(df):
    from datetime import datetime

    today = datetime.now().strftime("%d %b %Y")

    def parse_list(x):
        if isinstance(x, list):
            return x
        if isinstance(x, str):
            try:
                val = eval(x)
                if isinstance(val, list):
                    return val
            except:
                return [x]
        return []

    lines = []
    lines.append(f"Tanggal analisis: {today}\n")
    lines.append("Hasil screening saham berbasis aksi korporasi (IDX) + berita.\n")
    lines.append("Fokus pada sinyal yang berpotensi menggerakkan harga.\n")

    lines.append("\nDATA EMITEN:\n")

    for i, row in enumerate(df.head(10).itertuples(), 1):
        kode = getattr(row, "kode", "UNKNOWN")
        idx_score = getattr(row, "idx_score", getattr(row, "total_score", 0))
        news_score = getattr(row, "news_score", 0)
        combined = getattr(row, "combined", 0)

        # 🔥 FIX DI SINI
        judul_idx = parse_list(getattr(row, "judul_list_x", []))
        judul_news = parse_list(getattr(row, "judul_list_y", []))

        combined_titles = list(dict.fromkeys(judul_idx + judul_news))
        judul_ringkas = "; ".join(combined_titles[:4]) if combined_titles else "-"

        lines.append(
            f"{i}. {kode} | IDX:{idx_score} News:{news_score} Total:{combined}\n"
            f"   Sinyal: {judul_ringkas}\n"
        )

    # ======================
    # ANALYSIS TASK
    # ======================
    lines.append("\nTUGAS:\n")
    lines.append(
        "Identifikasi 3–5 saham paling menarik dari data di atas.\n"
        "Untuk masing-masing:\n"
        "- Jelaskan inti kejadian\n"
        "- Tentukan apakah katalis positif/negatif\n"
        "- Estimasi dampak ke harga (kecil/sedang/besar)\n"
        "- Tentukan horizon (short/medium/long term)\n"
        "- Berdasarkan harga sekarang apakah emiten ini layak diinvestasikan secara value? (undervalued/overvalued/fair)\n"
        "- Lakukan pencarian tambahan mengenai sektor emiten ini secara makro ekonomi!(misal harga komoditas sedang naik/turun) dan apa efeknya keperusahaan ini (profit/rugi/netral)?\n"
    )

    lines.append(
        "\nTambahkan:\n"
        "- Validasi fundamental singkat\n"
        "- Analisis teknikal ringkas\n"
        "- Rekomendasi: Buy / Spec Buy / Hold / Avoid (Urutkan dari Buy now sampai avoid)\n"
    )

    return "\n".join(lines)

In [27]:
prompt = build_master_prompt(final)

with open("prompt_final.txt", "w", encoding="utf-8") as f:
    f.write(prompt)

print("✅ Prompt siap digunakan")

✅ Prompt siap digunakan


---
# 📊 Kesimpulan & Dampak Bisnis (Business Impact)
Proyek ini membuktikan bagaimana otomasi data dapat secara drastis meningkatkan efisiensi operasional dalam proses riset investasi:
1. **Time Efficiency:** Memangkas waktu identifikasi *market mover* harian dari proses manual berjam-jam menjadi under 1 jam.
2. **Noise Reduction:** Berhasil menyaring ratusan berita harian menjadi Top 10 informasi paling krusial menggunakan sistem skor berbasis *rules*.
3. **Bridging the AI Gap:** Menyediakan *pipeline* yang menjembatani kelemahan AI (keterbatasan akses berita *real-time*) dengan memberikan *pre-processed context* berkualitas tinggi.

### 🚀 Future Roadmap
Untuk meningkatkan kapabilitas sistem, iterasi selanjutnya akan mencakup:
* Mengganti *Heuristic Keyword* dengan model **Transformer (IndoBERT)** untuk mendeteksi semantik berita secara lebih akurat.
* Mengintegrasikan data pergerakan harga historis (*Price Action*) untuk membangun **Backtesting Engine** guna menguji tingkat korelasi (*win rate*) dari skor berita.